# DNB MARC Record Fetching Workflow

This workflow automates the retrieval, parsing, and saving of bibliographic records from the German National Library (DNB) for a given list of titles and authors. It includes functions to query the DNB SRU interface, handle pagination, parse MARCXML data into structured fields (titles, authors, subjects, languages, publication info, etc.), recursively fetch related records, and incrementally save progress to a CSV file while avoiding duplicates.

In [ ]:
import os
import re
import time
import requests
import pandas as pd
import unicodedata
from bs4 import BeautifulSoup
from lxml import etree
import csv
from typing import Dict, List, Union

In [ ]:
titles_df = pd.read_csv('data/260504-meta-titles_OpenRefine.csv', sep=';', keep_default_na=False)

OUTPUT_FILE = 'data/dnb_results.csv'

In [ ]:
titles_df

## Functions

### SAVE & LOAD PROGRESS

#### `save_progress(df, output_file=OUTPUT_FILE)`
Saves progress to a CSV file, merging with existing data and avoiding duplicate records.

#### `load_done_ids(output_file=OUTPUT_FILE)`
Loads a list of already processed IDs from the saved CSV file to skip completed titles.

In [ ]:
def save_progress(df, output_file=OUTPUT_FILE):
    """
    Append or update progress in the output CSV file.
    Ensures records are unique by 'idn' (or full row if not available).
    """
    if "ID" not in df.columns:
        print("Warning: no 'ID' column in dataframe before saving.")
    
    # Combine with existing file if present
    if os.path.exists(output_file):
        existing = pd.read_csv(output_file, sep=";", quotechar='"', dtype=str)
        df = pd.concat([existing, df], ignore_index=True)
        if "idn" in df.columns:
            df = df.drop_duplicates(subset=["idn"], keep="last")
        else:
            df = df.drop_duplicates()

    # Ensure all values are stored as strings to avoid quoting issues
    df = df.astype(str)

    # Save file with all fields quoted and UTF-8 BOM for Excel compatibility
    df.to_csv(
        output_file,
        index=False,
        sep=";",
        quotechar='"',
        quoting=csv.QUOTE_ALL,
        encoding="utf-8-sig",
    )
    print(f"Progress saved: {len(df)} total records in {output_file}")


def load_done_ids(output_file=OUTPUT_FILE):
    """
    Load already processed IDs from the saved CSV file.
    Returns a set of IDs to skip in the next run.
    """
    if not os.path.exists(output_file):
        return set()
    df = pd.read_csv(output_file, sep=";", quotechar='"', dtype=str)
    if "wikiID" in df.columns:
        return set(df["wikiID"].dropna().unique())
    elif "idn" in df.columns:
        print("Warning: no 'wikiID' column in saved file; using 'idn' instead.")
        return set(df["idn"].dropna().unique())
    else:
        print("Warning: neither 'ID' nor 'idn' found in saved file.")
        return set()


### GND EXPORT

Fetch GND identifiers (Gemeinsame Normdatei, Wikidata property P227)
for one or more Wikidata items.

In [ ]:
WIKIDATA_API = "https://www.wikidata.org/w/api.php"
GND_PROPERTY = "P227"
# Wikimedia blocks requests with generic/default User-Agent strings (see
# https://meta.wikimedia.org/wiki/User-Agent_policy) — a descriptive one is required.
WIKIDATA_HEADERS = {
    "User-Agent": "DNB-GND-lookup-script/1.0 (research project; contact: your-email@example.com)"
}


def get_gnd_ids(
    wikidata_ids: Union[str, List[str]],
    include_deprecated: bool = False,
) -> Dict[str, List[str]]:
    if isinstance(wikidata_ids, str):
        wikidata_ids = [wikidata_ids]

    results: Dict[str, List[str]] = {}

    for i in range(0, len(wikidata_ids), 50):
        batch = wikidata_ids[i : i + 50]
        params = {
            "action": "wbgetentities",
            "ids": "|".join(batch),
            "props": "claims",
            "format": "json",
        }
        response = requests.get(WIKIDATA_API, params=params, headers=WIKIDATA_HEADERS, timeout=10)
        response.raise_for_status()
        data = response.json()

        for qid, entity in data.get("entities", {}).items():
            gnd_ids = []
            claims = entity.get("claims", {}).get(GND_PROPERTY, [])

            for claim in claims:
                if claim.get("rank") == "deprecated" and not include_deprecated:
                    continue
                value = claim.get("mainsnak", {}).get("datavalue", {}).get("value")
                if value:
                    gnd_ids.append(value)

            results[qid] = gnd_ids

    return results


def add_gnd_column(
    df: pd.DataFrame,
    id_column: str = "wikidata_id",
    new_column: str = "gnd_id",
    join_multiple: str = "; ",
    include_deprecated: bool = False,
) -> pd.DataFrame:
    df = df.copy()

    def _is_missing(qid):
        return pd.isna(qid) or not str(qid).strip()

    unique_ids = (
        df[id_column]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s != ""]
        .unique()
        .tolist()
    )
    lookup = get_gnd_ids(unique_ids, include_deprecated=include_deprecated)

    def _resolve(qid):
        if _is_missing(qid):
            return None
        qid = str(qid).strip()
        gnd_ids = lookup.get(qid, [])
        if not gnd_ids:
            return None
        if join_multiple is None:
            return gnd_ids
        return join_multiple.join(gnd_ids)

    df[new_column] = df[id_column].apply(_resolve)
    return df

def search_wikidata_person_ids(name, headers=WIKIDATA_HEADERS, sleep_sec=0.2, limit=5):
    """
    Search Wikidata for entities matching an author's name and return a list
    of candidate QIDs, ranked by Wikidata's own relevance ordering.

    Note: this is a name search, not an ID lookup, so it's inherently fuzzier
    than looking up a work via its wikiID — common names can match the wrong
    person. add_author_gnd_column() below reports which QID it picked for
    each author so you can spot-check ambiguous cases.
    """
    if not name or not str(name).strip():
        return []
    params = {
        "action": "wbsearchentities",
        "search": name,
        "language": "de",
        "type": "item",
        "limit": limit,
        "format": "json",
    }
    response = requests.get(WIKIDATA_API, params=params, headers=headers, timeout=10)
    response.raise_for_status()
    data = response.json()
    time.sleep(sleep_sec)
    return [r["id"] for r in data.get("search", [])]


def add_author_gnd_column(
    df: pd.DataFrame,
    firstname_col: str = "author_firstname",
    lastname_col: str = "author_lastname",
    gnd_column: str = "author_gnd_id",
    wikidata_column: str = "author_wikidata_id",
    include_deprecated: bool = False,
) -> pd.DataFrame:
    """
    Resolve a GND ID for each author by searching Wikidata by name (first
    candidate result that actually has a GND ID wins), then batch-fetching
    GND claims the same way add_gnd_column() does.

    Adds two columns:
      - `wikidata_column`: the QID Claude/the script matched to the author name
        (useful for spot-checking ambiguous or common names)
      - `gnd_column`: the resulting GND ID, if any

    Deduplicates by unique author name first, so a name repeated across many
    rows (e.g. Goethe appearing several times) is only looked up once.
    """
    df = df.copy()

    def _full_name(row):
        first = str(row.get(firstname_col, "") or "").strip()
        last = str(row.get(lastname_col, "") or "").strip()
        return f"{first} {last}".strip()

    author_names = df.apply(_full_name, axis=1)
    unique_names = [n for n in author_names.unique() if n]

    # Step 1: search Wikidata by name for each unique author -> candidate QIDs
    name_to_candidates = {name: search_wikidata_person_ids(name) for name in unique_names}

    # Step 2: batch-fetch GND claims for every candidate QID across all names
    all_candidate_ids = sorted({qid for c in name_to_candidates.values() for qid in c})
    gnd_lookup = get_gnd_ids(all_candidate_ids, include_deprecated=include_deprecated) if all_candidate_ids else {}

    # Step 3: for each name, take the first candidate that actually has a GND ID
    name_to_result = {}
    for name, candidates in name_to_candidates.items():
        chosen_qid, chosen_gnd = None, None
        for qid in candidates:
            ids = gnd_lookup.get(qid, [])
            if ids:
                chosen_qid, chosen_gnd = qid, ids[0]
                break
        name_to_result[name] = (chosen_qid, chosen_gnd)

    df[wikidata_column] = author_names.map(lambda n: name_to_result.get(n, (None, None))[0])
    df[gnd_column] = author_names.map(lambda n: name_to_result.get(n, (None, None))[1])
    return df


In [ ]:
# Enrich titles_df with a GND ID for each literary text, looked up via its Wikidata ID.
# The OpenRefine export uses the sentinel string "noWikiID" for titles with no Wikidata
# match — treat that the same as a missing ID before looking anything up.
titles_df["wikiID"] = titles_df["wikiID"].replace("noWikiID", pd.NA)

titles_df = add_gnd_column(
    titles_df,
    id_column="wikiID",
    new_column="title_gnd_id",
)

missing_titles = titles_df["title_gnd_id"].isna().sum()
print(f"{missing_titles} of {len(titles_df)} titles have no GND ID available via Wikidata.")

# Also resolve a GND ID for each author, via a Wikidata name search (no author
# QID column exists in this dataset, so this is a name match rather than an ID
# lookup — check author_wikidata_id for common/ambiguous names before trusting it).
titles_df = add_author_gnd_column(titles_df)

missing_authors = titles_df["author_gnd_id"].isna().sum()
print(f"{missing_authors} of {len(titles_df)} authors have no GND ID resolved via Wikidata name search.")

titles_df[["author_firstname", "author_lastname", "author_wikidata_id", "author_gnd_id"]].drop_duplicates()


In [ ]:
titles_df

### DNB SRU FUNCTIONS

#### `dnb_sru_all(title, author=None, batch_size=50, sleep_sec=2, verbose=True)`
Queries the DNB SRU interface for bibliographic records by title and optional author, handling pagination and retries.

#### `dnb_sru_all_by_idn(idn, sleep_sec=0.2)`
Fetches a single bibliographic record from the DNB using its unique IDN identifier.

In [ ]:
def dnb_sru_all(title, author=None, gnd_id=None, author_gnd_id=None, batch_size=50, sleep_sec=2, verbose=True):
    """
    Query the DNB SRU interface for a given title (and optional author).

    Tries queries in order of precision, stopping at the first that returns
    results:
      1. the work's own GND ID, via the "dnb.nid" (Normdatenidentifier) index
      2. the author's GND ID (from a Wikidata name search) combined with the
         title, via "dnb.aunidref" ("Autor-Referenz per NID")
      3. free-text title + author
      4. free-text title only (fallback)

    Fetches results in batches and follows SRU pagination.
    """
    base_url = "https://services.dnb.de/sru/dnb"
    title_clean = title.replace("'", "\\'")
    all_records = []

    # Build queries, most precise first (see docstring for the order)
    queries = []
    if gnd_id:
        queries.append(f'dnb.nid="{gnd_id}"')
    if author_gnd_id:
        queries.append(f'dnb.aunidref="{author_gnd_id}" and (dc.title all "{title_clean}")')
    if author:
        queries.append(f'(dc.title all "{title_clean}") and (dc.creator all "{author}")')    
    queries.append(f'(dc.title all "{title_clean}")')  # fallback

    # Loop through queries in order of precision until results found
    for q in queries:
        start = 1
        total_found = 0
        if verbose:
            print(f"Trying query: {q}")

        while True:
            params = {
                "version": "1.1",
                "operation": "searchRetrieve",
                "recordSchema": "MARC21-xml",
                "maximumRecords": str(batch_size),
                "startRecord": str(start),
                "query": q
            }

            # Query DNB API and handle potential errors
            try:
                r = requests.get(base_url, params=params, timeout=30)
                r.raise_for_status()
            except requests.exceptions.RequestException as e:
                print(f"Request failed at start={start}: {e}")
                break

            xml = BeautifulSoup(r.content, "xml")
            records = xml.find_all("record", {"type": "Bibliographic"})
            num = len(records)

            if verbose:
                print(f" → Retrieved {num} records (start={start})")

            if num == 0:
                break  # no more results

            all_records.extend(records)
            total_found += num

            # Handle pagination if more results available
            next_pos_tag = xml.find("nextRecordPosition")
            if next_pos_tag and next_pos_tag.text:
                start = int(next_pos_tag.text)
                time.sleep(sleep_sec)
            else:
                break

        # Stop after first query that returns results
        if total_found > 0:
            if verbose:
                print(f"Found {total_found} records for query: {q}")
            break
        else:
            if verbose:
                print(f"No results for query: {q}")

    return all_records


def dnb_sru_all_by_idn(idn, sleep_sec=0.2):
    """
    Fetch a specific record (and possible variants) from DNB by its IDN.
    Used to retrieve related records recursively.
    """
    base_url = "https://services.dnb.de/sru/dnb"
    query = f"idn={idn}"
    params = {
        "version": "1.1",
        "operation": "searchRetrieve",
        "recordSchema": "MARC21-xml",
        "maximumRecords": "10",
        "query": query
    }
    try:
        r = requests.get(base_url, params=params, timeout=30)
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"IDN fetch failed for {idn}: {e}")
        return []
    xml = BeautifulSoup(r.content, "xml")
    records = xml.find_all("record", {"type": "Bibliographic"})
    time.sleep(sleep_sec)
    return records


### PARSE MARC XML

#### `parse_record(record)`
Parses MARCXML data into a structured dictionary, extracting fields like title, author, subjects, languages, and publication info.

In [ ]:
def parse_record(record):
    """
    Parse one MARCXML record and extract key bibliographic fields.
    Includes both standard and additional (subject, genre, etc.) fields.
    """
    record_str = unicodedata.normalize("NFC", str(record))
    ns = {"marc": "http://www.loc.gov/MARC21/slim"}
    try:
        xml = etree.fromstring(record_str.encode("utf-8"))
    except Exception:
        return {}

    # --- Helper functions for MARC structure ---
    def get_control(tag):
        nodes = xml.xpath(f"marc:controlfield[@tag='{tag}']", namespaces=ns)
        return nodes[0].text if nodes else None

    def get_datafield(tag, code=None):
        nodes = xml.xpath(f"marc:datafield[@tag='{tag}']", namespaces=ns)
        texts = []
        for node in nodes:
            if code:
                subs = node.xpath(f"marc:subfield[@code='{code}']", namespaces=ns)
                texts.extend([s.text.strip() for s in subs if s.text])
            else:
                texts.append(" || ".join([s.text.strip() for s in node.xpath("marc:subfield", namespaces=ns) if s.text]))
        return " || ".join(texts) if texts else None

    def get_fields(tag_list):
        """Combine multiple MARC tags into one string."""
        combined = []
        for t in tag_list:
            val = get_datafield(t)
            if val:
                combined.append(val)
        return " || ".join(combined) if combined else None

    # --- Control fields ---
    idn = get_control("001")
    field_008 = get_control("008")

    # --- Title and variant titles ---
    title = get_datafield("245", "a") or get_datafield("240", "a") or get_datafield("130", "a")
    standardised_title = get_datafield("240", "a") or get_datafield("130", "a") or title
    alternative_titles = get_datafield("246")

    # --- Author and identifiers ---
    author = get_datafield("100", "a") or get_datafield("700", "a")
    rela = get_datafield("100", "e")
    gnd = get_datafield("100", "0") or get_datafield("700", "0")

    # --- Publication info ---
    pub_year = get_datafield("264", "c") or (field_008[7:11] if field_008 else None)

    # --- Links, notes, and series ---
    links = get_datafield("856", "u")
    series = get_fields(["490", "830"])
    notes = get_fields([f"{i:03}" for i in range(500, 600)])
    related_items = get_fields(["776", "787"])

    # --- Additional MARC fields ---
    language_codes = get_datafield("041")
    subject_chron = get_datafield("648")    # chronological subject
    subject_topical = get_datafield("650")  # topical subject
    genre_form = get_datafield("655")       # genre/form
    publication_info = get_fields(["264"])  # full publication statement

    return {
        "idn": idn,
        "title": title,
        "standardised_title": standardised_title,
        "alternative_titles": alternative_titles,
        "author": author,
        "rela": rela,
        "gnd": gnd,
        "pub_year": pub_year,
        "links": links,
        "series": series,
        "notes": notes,
        "related_items": related_items,
        "language_codes": language_codes,
        "subject_chron": subject_chron,
        "subject_topical": subject_topical,
        "genre_form": genre_form,
        "publication_info": publication_info,
        "control_008": field_008
    }


### RECURSIVE FETCHING

#### `extract_idns_from_related(text)`
Extracts IDN identifiers from related-item fields in MARCXML using regular expressions.

#### `fetch_related_records(record_ids, seen=None)`
Retrieves and parses related DNB records recursively, avoiding duplicates through a `seen` set.

In [ ]:
def extract_idns_from_related(text):
    """Extract numeric DNB IDNs from a text string using regex."""
    if not text:
        return []
    return re.findall(r'\b\d{8,}\b', text)


def fetch_related_records(record_ids, seen=None):
    """
    Recursively fetch and parse records linked to the given IDNs.
    Uses a 'seen' set to avoid infinite loops.
    """
    if seen is None:
        seen = set()
    all_parsed = []
    for rid in record_ids:
        if rid in seen:
            continue
        seen.add(rid)
        recs = dnb_sru_all_by_idn(rid)
        for r in recs:
            parsed = parse_record(r)
            if parsed:
                all_parsed.append(parsed)
                related_ids = extract_idns_from_related(parsed.get("related_items"))
                # Recursively fetch related records
                all_parsed.extend(fetch_related_records(related_ids, seen))
    return all_parsed
    

### MAIN SCRIPT

#### `main(titles_df)`
Iterates through the input dataframe of titles, fetches main and related DNB records for each, and saves results incrementally.

In [ ]:
def main(titles_df):
    done_ids = load_done_ids()
    print(f"Already done: {len(done_ids)} titles → {sorted(done_ids)}")

    try:
        for _, row in titles_df.iterrows():
            ID = row["wikiID"]
            if ID in done_ids:
                print(f"Skipping already done: {row['title_original']}")
                continue

            title = row["title"]

            # --- Clean and combine author name ---
            first = str(row.get("author_firstname", "") or "").strip()
            last = str(row.get("author_lastname", "") or "").strip()
            author = f"{first} {last}".strip() or None

            # --- GND IDs resolved earlier via Wikidata ---
            gnd_id = row.get("title_gnd_id")
            gnd_id = gnd_id if pd.notna(gnd_id) else None

            author_gnd_id = row.get("author_gnd_id")
            author_gnd_id = author_gnd_id if pd.notna(author_gnd_id) else None

            gnd_note = ""
            if gnd_id:
                gnd_note += f" [title GND: {gnd_id}]"
            if author_gnd_id:
                gnd_note += f" [author GND: {author_gnd_id}]"

            print(f"\nFetching main records for: {title}" + (f" ({author})" if author else "") + gnd_note)

            # --- Fetch main records ---
            main_records = dnb_sru_all(title, author=author, gnd_id=gnd_id, author_gnd_id=author_gnd_id, batch_size=25)
            parsed = [parse_record(r) for r in main_records if r]
            print(f"Retrieved {len(parsed)} main records for {title}")

            for p in parsed:
                p.update(row.to_dict())  # attach metadata from input CSV

            # --- Fetch related records ---
            related_idns = list(set(idn for p in parsed for idn in extract_idns_from_related(p.get("related_items"))))
            print(f"Fetching {len(related_idns)} related records for {title}")

            related_parsed = fetch_related_records(related_idns)
            for rp in related_parsed:
                rp.update(row.to_dict())
            print(f"Retrieved {len(related_parsed)} related records for {title}")

            # --- Save progress after each title ---
            df_title = pd.DataFrame(parsed + related_parsed)
            if not df_title.empty:
                save_progress(df_title)
            else:
                print(f"No records found for {title}")

    except KeyboardInterrupt:
        print("\nManual stop detected — progress saved.")
    except Exception as e:
        print(f"\nError occurred: {e}")
    finally:
        print("Script finished safely.")


### Execution

In [ ]:
titles_df

In [ ]:
main(titles_df)